# Case Study: Text-to-SQL Accuracy Estimation on Spider 1.0

Spider 1.0 is a large-scale text-to-SQL benchmark covering 200 relational databases across diverse
domains. Each example pairs a natural language question with a gold SQL query.

This case study estimates the accuracy of a Claude Haiku text-to-SQL generator on the Spider
training split. Human evaluation is conceptually straightforward: execute both the gold SQL and
the predicted SQL against the corresponding SQLite database, then compare the result sets.
In practice, running this at scale with human reviewers is expensive.

A cheaper alternative is an LLM judge: given the question, the database schema, and the predicted
SQL, the judge decides whether the translation is correct. The judge is not perfectly calibrated.
Its reliability varies across databases: a simple, well-structured schema is easier to judge
correctly than one with many tables, unusual naming conventions, or domain-specific vocabulary.
Each database therefore forms a natural stratum.

GLIDE applies Stratified PPI++ to correct for the per-database judge bias and produce a valid
confidence interval with far fewer human annotations than a purely classical approach would require.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from glide.estimators import (
    ClassicalMeanEstimator,
    StratifiedClassicalMeanEstimator,
    StratifiedPPIMeanEstimator,
)
from glide.samplers import StratifiedSampler
from glide.scientific_validation import compute_hits, coverage_with_error_bar, run_monte_carlo
from glide.simulators import generate_binary_dataset, simulate_annotation

plt.rcParams.update(
    {
        "font.size": 18,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "legend.fontsize": 16,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.titlesize": 19,
    }
)

In [ ]:
RANDOM_SEED = 42
N_LABELED = 400
CONFIDENCE_LEVEL = 0.95
N_SEEDS = 1000

# HF_REPO = "glide-py/spider-text-to-sql"

# dataset = load_dataset(HF_REPO, split="train")
ds_path = "../spider/data/spider_dataset.parquet"

dataset = pd.read_parquet(ds_path)

groups = np.array(dataset["db_id"])

group_ids = {grp: i for i, grp in enumerate(set(groups))}
kept_group_ids = [i for i, nb in enumerate(np.bincount([group_ids[grp] for grp in groups])) if nb > 50]

dataset = dataset[dataset["db_id"].apply(lambda x: group_ids[x] in kept_group_ids)]

y_true_oracle = np.array(dataset["human_label"], dtype=float)
y_proxy = np.array(dataset["llm_judge_label"], dtype=float)
groups = np.array(dataset["db_id"])

y_true_oracle, y_proxy = generate_binary_dataset(len(groups), random_seed=RANDOM_SEED)  #######

In [ ]:
true_mean = np.mean(y_true_oracle)

WORKFLOW_ESTIMATORS = {
    "LLM Judge": ClassicalMeanEstimator,
    "Human sample": StratifiedClassicalMeanEstimator,
    "GLIDE": StratifiedPPIMeanEstimator,
}

COLORS = {
    "LLM Judge": "red",
    "Human sample": "steelblue",
    "GLIDE": "purple",
}

## The Human Annotator

`y_true_oracle` was produced by executing both the gold SQL and the predicted SQL against the
corresponding Spider SQLite database, sorting each result set, and comparing them as Python lists.
An execution error scores 0; an exact result set match scores 1.
This process is deterministic and fully reproducible.

The full dataset and the scripts used to build it are published at
[glide-py/spider-text-to-sql](https://huggingface.co/datasets/glide-py/spider-text-to-sql)
on HuggingFace. In a real deployment, the oracle role would be played by human reviewers
on a sampled subset. Here, the programmatic comparison gives us a complete oracle for all
examples, which we use to simulate the annotation process with `simulate_annotation`.

In [ ]:
df = pd.DataFrame(
    {
        "db_id": groups,
        "human_label": y_true_oracle,
        "llm_judge_label": y_proxy,
    }
)
summary = df.groupby("db_id")[["human_label", "llm_judge_label"]].mean()

x = np.arange(len(summary))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(
    x - width / 2, summary["human_label"], width, label="Human (ground truth)", color=COLORS["Human sample"], alpha=0.75
)
ax.bar(
    x + width / 2, summary["llm_judge_label"], width, label="LLM Judge (proxy)", color=COLORS["LLM Judge"], alpha=0.75
)
ax.set_xticks(x)
ax.set_xticklabels(summary.index, rotation=45, ha="right")
ax.set_ylabel("Mean accuracy")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
db_ids, db_counts = np.unique(groups, return_counts=True)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(db_ids, db_counts, color="steelblue", alpha=0.75)
ax.set_xticks(range(len(db_ids)))
ax.set_xticklabels(db_ids, rotation=45, ha="right")
ax.set_ylabel("Number of examples")
plt.tight_layout()
plt.show()

## Stratified Sampling

Annotating every example with a human reviewer would be expensive. GLIDE draws a fixed budget
using Neyman allocation, which directs more budget to databases where the proxy is least reliable:
databases with higher variance in judge accuracy receive proportionally more of the annotation budget.

In [ ]:
xi = StratifiedSampler().sample(y_proxy, groups, n_samples=N_LABELED, strategy="neyman", random_seed=RANDOM_SEED)
y_true = simulate_annotation(y_true_oracle, xi)
print(f"Annotated: {int(xi.sum())} / {len(xi)} examples")

In [ ]:
result_judge = ClassicalMeanEstimator().estimate(
    y_proxy,
    metric_name="SQL Accuracy",
    confidence_level=CONFIDENCE_LEVEL,
)
result_human = StratifiedClassicalMeanEstimator().estimate(
    y_true,
    groups,
    metric_name="SQL Accuracy",
    confidence_level=CONFIDENCE_LEVEL,
)
result_glide = StratifiedPPIMeanEstimator().estimate(
    y_true,
    y_proxy,
    groups,
    metric_name="SQL Accuracy",
    confidence_level=CONFIDENCE_LEVEL,
    power_tuning=True,
)
print(result_glide.summary())

In [ ]:
results = [result_judge, result_human, result_glide]
labels = [
    f"LLM Judge\n({len(y_proxy)} examples)",
    f"Human sample\n({int(xi.sum())} examples)",
    f"GLIDE\n({int(xi.sum())} human + {len(y_proxy)} proxy)",
]
workflow_names = list(WORKFLOW_ESTIMATORS.keys())
y_pos = [2, 1, 0]

fig, ax = plt.subplots(figsize=(11, 5.5))
for y, (workflow, result, label) in zip(y_pos, zip(workflow_names, results, labels)):
    ci = result.confidence_interval
    color = COLORS[workflow]
    ax.plot([ci.lower_bound, ci.upper_bound], [y, y], color=color, linewidth=4, solid_capstyle="round", zorder=3)
    for xc in [ci.lower_bound, ci.upper_bound]:
        ax.plot([xc, xc], [y - 0.2, y + 0.2], color=color, linewidth=2.5, zorder=3)
    ax.scatter(result.mean, y, s=200, color=color, zorder=5, edgecolors="white", linewidths=2.5)
    ax.text(result.mean, y + 0.25, f"{result.mean:.1%}", ha="center", va="bottom", color=color, fontweight="bold")
    ax.text(
        result.mean, y - 0.25, f"[{ci.lower_bound:.1%},  {ci.upper_bound:.1%}]", ha="center", va="top", color="#888888"
    )

ax.axvline(true_mean, color="black", linestyle="--", linewidth=2.5, zorder=4)
ax.text(true_mean + 0.004, 2.72, f"True accuracy  {true_mean:.1%}", color="black", fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("SQL Accuracy")
ax.set_xlim(None, None)
ax.set_ylim(-0.8, 3.2)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

## Coverage Validity

A confidence interval is **valid** if it contains the true parameter at the stated rate.
A 95% interval is valid when, across many independent draws of the sampling mask, roughly 95%
of the resulting intervals contain the true mean.

We run a Monte Carlo experiment sweeping the confidence level from 0.55 to 0.95 and check
that each workflow's empirical coverage tracks the diagonal. The `LLM Judge` workflow is
expected to fall well below because its point estimate is biased.
See the [Scientific Validation Methodology](scientific_validation.md) page for more details.

In [ ]:
def preprocess_data(y_true_oracle, y_proxy, groups, workflow, seed):
    if workflow == "LLM Judge":
        return (y_proxy,)
    elif workflow == "Human sample":
        xi = StratifiedSampler().sample(y_proxy, groups, n_samples=N_LABELED, strategy="neyman", random_seed=seed)
        y_true = simulate_annotation(y_true_oracle, xi)
        return y_true, groups
    elif workflow == "GLIDE":
        xi = StratifiedSampler().sample(y_proxy, groups, n_samples=N_LABELED, strategy="neyman", random_seed=seed)
        y_true = simulate_annotation(y_true_oracle, xi)
        return y_true, y_proxy, groups
    else:
        raise ValueError(f"Unknown workflow: {workflow}")


def generate_estimates(y_true_oracle, y_proxy, groups, seed):
    estimates = {}
    for workflow, EstimatorClass in WORKFLOW_ESTIMATORS.items():
        estimator = EstimatorClass()
        data = preprocess_data(y_true_oracle, y_proxy, groups, workflow, seed)
        kwargs = {"confidence_level": CONFIDENCE_LEVEL}
        if workflow == "GLIDE":
            kwargs["power_tuning"] = True
        result = estimator.estimate(*data, **kwargs)
        estimates[workflow] = {
            "mean": result.mean,
            "std": result.std,
            "confidence_interval": result.confidence_interval,
            "effective_sample_size": getattr(result, "effective_sample_size", None),
        }
    return estimates

In [ ]:
confidence_levels = np.round(np.arange(0.55, 1.00, 0.05), 2)

raw_stats = run_monte_carlo(
    confidence_levels,
    run_seed=lambda seed: generate_estimates(y_true_oracle, y_proxy, groups, seed),
    n_seeds=N_SEEDS,
)

coverages_by_confidence_level = {}
for cl in confidence_levels:
    hits = compute_hits(raw_stats, cl, true_mean)
    coverages_by_confidence_level[cl] = {
        workflow: coverage_with_error_bar(hits[workflow], confidence_level=CONFIDENCE_LEVEL)
        for workflow in WORKFLOW_ESTIMATORS
    }

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(confidence_levels, confidence_levels, color="black", lw=1.5, linestyle="--", label="Ideal")

for workflow in WORKFLOW_ESTIMATORS:
    mean_ci = np.array([coverages_by_confidence_level[cl][workflow] for cl in confidence_levels])
    ax.plot(confidence_levels, mean_ci[:, 0], marker="o", color=COLORS[workflow], label=workflow)
    ax.fill_between(confidence_levels, mean_ci[:, 1], mean_ci[:, 2], alpha=0.15, color=COLORS[workflow])

ax.set_xlabel("Target confidence level")
ax.set_ylabel("Observed coverage")
ax.set_xlim(0.5, 1.0)
ax.set_ylim(0.5, 1.0)
ax.legend()
plt.tight_layout()
plt.show()

Both `Human sample` and `GLIDE` achieve valid coverage, tracking the diagonal across all tested
confidence levels. `LLM Judge` falls below because its point estimate is biased by the judge's
systematic errors, causing intervals centered at the wrong value to miss the true mean more
often than the nominal rate allows.

---

## Confidence Interval Width

Both `Human sample` and `GLIDE` use the same `N_LABELED` annotation budget, so differences
in interval width reflect the information gain from incorporating the full proxy signal.
We report the mean width and a percentile band across Monte Carlo seeds.

In [ ]:
width_by_confidence_level = {}
for cl in confidence_levels:
    width_by_confidence_level[cl] = {}
    for workflow in WORKFLOW_ESTIMATORS:
        lower = raw_stats[workflow]["lower_bounds"][cl]
        upper = raw_stats[workflow]["upper_bounds"][cl]
        width_by_confidence_level[cl][workflow] = upper - lower

lower_percentile = round(((1 - CONFIDENCE_LEVEL) / 2) * 100)
upper_percentile = 100 - lower_percentile

fig, ax = plt.subplots(figsize=(9, 5))

for workflow in WORKFLOW_ESTIMATORS:
    means_w = [np.mean(width_by_confidence_level[cl][workflow]) for cl in confidence_levels]
    q_lower = [np.percentile(width_by_confidence_level[cl][workflow], lower_percentile) for cl in confidence_levels]
    q_upper = [np.percentile(width_by_confidence_level[cl][workflow], upper_percentile) for cl in confidence_levels]
    ax.plot(confidence_levels, means_w, marker="o", color=COLORS[workflow], label=workflow)
    ax.fill_between(confidence_levels, q_lower, q_upper, alpha=0.15, color=COLORS[workflow])

ax.set_xlabel("Confidence level")
ax.set_ylabel("Confidence interval width")
ax.set_xlim(0.5, 1.0)
ax.yaxis.set_ticks(ax.get_yticks()[1:-1:2])
ax.legend()
plt.tight_layout()
plt.show()

GLIDE achieves narrower confidence intervals than `Human sample` at the same annotation
budget, driven by the information extracted from the full set of proxy labels.
The `LLM Judge` intervals appear narrow but are misleading given that they do not achieve
valid coverage.

## Results

The forest plot illustrates three estimates of text-to-SQL accuracy on Spider 1.0.

**LLM Judge.** The judge estimate uses all proxy labels without bias correction.
The gap between the judge estimate and the true accuracy (dashed line) reveals the direction
and magnitude of the judge's systematic error on this model.

**Human sample.** The stratified classical estimate from the annotated subset is unbiased:
it centers near the true accuracy. Its confidence interval is wide, however, because the
annotated subset is a small fraction of the full dataset.

**GLIDE.** The stratified PPI++ estimate is also unbiased by construction but achieves a
narrower confidence interval by leveraging all proxy labels. The width reduction reflects
the effective sample size gain from combining human annotations with the proxy signal.

The per-database breakdown in the bias chart above reveals why stratification improves over
plain PPI++: databases where the judge is most miscalibrated receive a larger share of the
annotation budget under Neyman allocation, targeting human effort where the proxy is least reliable.

---

## Annex: Dataset and Pipeline

The labeled dataset used in this notebook is published on Hugging Face at
[glide-py/spider-text-to-sql](https://huggingface.co/datasets/glide-py/spider-text-to-sql)
together with the scripts and a `README.md` that explain how to reproduce the dataset from scratch.

### Spider 1.0

Spider 1.0 is a large-scale text-to-SQL benchmark containing 7,000 training examples
spread across 166 databases in diverse domains (sports, finance, government, entertainment, and more).
Each example pairs a natural language question with a gold SQL query.
We use the official Spider 1.0 release (not the HuggingFace `xlangai/spider` snapshot)
to obtain the SQLite database files required for SQL execution and the `tables.json` schema file
required for prompts.

### Strata

Each database (`db_id`) is a natural stratum: examples within a database share the same schema,
domain vocabulary, and SQL patterns, while the judge's reliability varies across databases for
the same reasons. The exploration script (`scripts/explore_dataset.py`) was used to decide the
grouping strategy and annotation budget before committing to the data pipeline.

### SQL Generator

SQL predictions were generated using **Claude Haiku 4.5** (`claude-haiku-4-5-20251001`) via
the Anthropic API, called at temperature 0 with a 256-token output limit.

Each call uses a fixed system prompt:

```
You are an expert SQL writer. Given a natural language question and a database schema,
write the SQL query that answers the question. Return ONLY the SQL query, no explanation.
```

The user turn includes the full database schema (all tables, columns, primary keys, and foreign keys
derived from `tables.json`) alongside the question:

```
Database: {db_id}

Schema:
{schema}

Question: {question}

Return your SQL query:
```

Including the full schema prevents the model from inventing column or table names that do not
exist in the database.

### LLM-as-Judge

Proxy labels were generated using **Claude Sonnet 4.6** (`claude-sonnet-4-6`) via the Anthropic API,
also called at temperature 0. The judge receives the schema in its prompt so it can reason about
whether the predicted SQL is structurally valid for the given database, not just whether it looks
plausible in isolation.

System prompt:

```
You are evaluating SQL queries generated by an AI assistant.
Given a natural language question, the database schema, and a predicted SQL query,
determine whether the predicted SQL correctly answers the question on this database.
Be rigorous: pay attention to missing WHERE clauses, wrong aggregations, incorrect JOINs,
and column or table names that do not exist in the schema.
Your response must be a single digit: 0 or 1. Output nothing else.
```

User turn:

```
Database: {db_id}

Schema:
{schema}

Question: {question}
Predicted SQL: {predicted_sql}

Output 1 if the SQL correctly answers the question, 0 if it does not. Output only 0 or 1
and absolutely nothing more.
```

The model response is parsed by looking for a bare `0` or `1` in the stripped output.

### Human Label Generation

Human labels are produced programmatically with no LLM involved: both the gold SQL and the
predicted SQL are executed against the corresponding SQLite database file; the result sets are
sorted and compared as Python lists. An execution error in the predicted query scores 0;
an exact result-set match scores 1. The comparison is order-insensitive because SQL does not
guarantee row ordering. The database connection is closed after every example to avoid file locking.
This process is deterministic, free, and fully reproducible given the original Spider SQLite files.